# Phase 4: Project Development — Construction Cost Overrun Prediction

**Project:** Industry AI Flow — AI-Powered Construction Industry Platform

**Program:** Integrated Artificial Intelligence, SAIT Capstone Project

**Team Members:**
- Angel Daniel Bustamante Perez
- Jason Niu
- Jack Si

**Instructor:** Reeta

**Date:** March 2026

---

This notebook follows the **CRISP-DM (Cross-Industry Standard Process for Data Mining)** methodology to develop a machine learning model that predicts construction project cost overruns.

## 1. Business Understanding

### 1.1 Problem Description

Construction projects frequently exceed their original budgets, causing financial strain for developers, contractors, and stakeholders. In Canada, cost overruns are a common challenge across residential, commercial, infrastructure, and institutional construction projects. These overruns are influenced by many factors including project complexity, contractor experience, weather conditions, material price volatility, and the number of change orders during construction.

### 1.2 Business Context and Objectives

Our project, **Industry AI Flow**, is an AI-powered platform designed to assist the construction industry. One of its three core capabilities is **Construction Cost Estimation** — a machine learning model that predicts the percentage of cost overrun for a given project based on its characteristics.

**Business Objectives:**
- Provide construction professionals with early warnings about potential budget overruns
- Identify the key factors that drive cost overruns so that project managers can take preventive action
- Deliver predictions with confidence intervals so that stakeholders can plan contingency budgets

### 1.3 Impact of Solving This Problem

An accurate cost overrun prediction model enables:
- **Better budgeting:** Project managers can set more realistic budgets by factoring in predicted overruns
- **Risk mitigation:** Identifying high-risk factors (e.g., too many change orders, low contractor rating) allows proactive intervention
- **Decision support:** Stakeholders can compare project plans and choose options with lower predicted overrun risk
- **Industry value:** Construction is one of the largest industries in Canada, and even a small percentage improvement in cost estimation accuracy can save millions of dollars across the industry

---
## 2. Data Collection and Data Understanding

### 2.1 Data Collection

Our dataset was curated by our team member **Jack Si**, who has a construction industry background. The dataset contains **10,000 construction project records** with 25 features covering project specifications, contractor information, risk factors, and actual outcomes.

The data is stored as a CSV file: `unified_construction_projects_enhanced.csv`

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')

In [ ]:
# Load the dataset
# Use pathlib to resolve the path relative to the notebook location
from pathlib import Path

dataset_path = Path(__file__).parent / '../../datasets/unified_construction_projects_enhanced.csv' \
    if '__file__' in dir() else Path('../../datasets/unified_construction_projects_enhanced.csv')

# Try relative path first, then absolute path
import os
candidates = [
    '../../datasets/unified_construction_projects_enhanced.csv',
    os.path.join(os.getcwd(), 'datasets/unified_construction_projects_enhanced.csv'),
    '/Users/openclaw/Documents/github.com/Industry-AI-Flow/datasets/unified_construction_projects_enhanced.csv',
]
for p in candidates:
    if os.path.exists(p):
        dataset_path = p
        break

df = pd.read_csv(dataset_path)
print(f'Dataset loaded from: {dataset_path}')
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

### 2.2 Non-Visual EDA

We start by examining data types, summary statistics, and missing values.

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
# Check for duplicate rows
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

In [ ]:
# Unique values for categorical columns
cat_cols = ['project_type', 'location', 'on_budget', 'on_schedule', 'data_source']
for col in cat_cols:
    print(f'\n{col} ({df[col].nunique()} unique):')
    print(df[col].value_counts())

### 2.3 Dependent and Independent Variables

- **Dependent variable (target):** `cost_overrun_pct` — the percentage by which the actual cost exceeded (or was under) the estimated cost
- **Independent variables (features):** Project characteristics including `sqft`, `floors`, `num_units`, `planned_duration_weeks`, `estimated_cost_cad`, `contractor_rating`, `complexity_score`, `team_experience_years`, `num_change_orders`, `weather_risk_factor`, `material_volatility`, `num_subcontractors`, `budget_pressure`, `risk_score`, `risk_score_original`, plus categorical features `project_type` and `location`
- **Excluded columns:** `project_id` (identifier), `actual_cost_cad` and `actual_duration_weeks` (post-hoc outcomes — using them would be data leakage), `schedule_delay_pct`, `on_budget`, `on_schedule` (derived from actuals), `data_source` (constant)

In [ ]:
# Define target and feature columns
target = 'cost_overrun_pct'

# Columns to exclude (identifiers, post-hoc outcomes, constants)
exclude_cols = [
    'project_id', 'actual_cost_cad', 'actual_duration_weeks',
    'schedule_delay_pct', 'on_budget', 'on_schedule', 'data_source', target
]

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f'Target: {target}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

### 2.4 Visual EDA — Univariate Analysis

We examine the distribution of each variable individually.

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[target], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Cost Overrun (%)')
axes[0].set_xlabel('Cost Overrun %')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df[target].mean(), color='red', linestyle='--', label=f'Mean: {df[target].mean():.1f}%')
axes[0].legend()

axes[1].boxplot(df[target], vert=True)
axes[1].set_title('Box Plot of Cost Overrun (%)')
axes[1].set_ylabel('Cost Overrun %')

plt.tight_layout()
plt.show()

print(f'Mean: {df[target].mean():.2f}%, Median: {df[target].median():.2f}%, Std: {df[target].std():.2f}%')
print(f'Skewness: {df[target].skew():.3f}, Kurtosis: {df[target].kurtosis():.3f}')

In [ ]:
# Distributions of key numeric features
num_features = [
    'sqft', 'floors', 'estimated_cost_cad', 'contractor_rating',
    'complexity_score', 'team_experience_years', 'num_change_orders',
    'risk_score', 'weather_risk_factor', 'material_volatility'
]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Distributions of Key Numeric Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of categorical features
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['project_type'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Projects by Type')
axes[0].set_xlabel('Count')

df['location'].value_counts().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Projects by Location')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

### 2.5 Visual EDA — Bivariate Analysis

We examine the relationship between each key feature and the target variable.

In [ ]:
# Scatter plots: key features vs. cost overrun
scatter_features = [
    'num_change_orders', 'risk_score', 'contractor_rating',
    'team_experience_years', 'complexity_score', 'estimated_cost_cad'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(scatter_features):
    axes[i].scatter(df[col], df[target], alpha=0.15, s=10, color='steelblue')
    # Add trend line
    z = np.polyfit(df[col], df[target], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Cost Overrun %')
    corr = df[col].corr(df[target])
    axes[i].set_title(f'{col} (r = {corr:.3f})')

plt.suptitle('Feature vs. Cost Overrun (%) with Trend Lines', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: cost overrun by project type and location
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

df.boxplot(column=target, by='project_type', ax=axes[0], vert=True, patch_artist=True)
axes[0].set_title('Cost Overrun by Project Type')
axes[0].set_xlabel('Project Type')
axes[0].set_ylabel('Cost Overrun %')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
plt.sca(axes[0])
plt.title('Cost Overrun by Project Type')

df.boxplot(column=target, by='location', ax=axes[1], vert=True, patch_artist=True)
axes[1].set_title('Cost Overrun by Location')
axes[1].set_xlabel('Location')
axes[1].set_ylabel('Cost Overrun %')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
plt.sca(axes[1])
plt.title('Cost Overrun by Location')

plt.suptitle('')  # Remove auto-generated suptitle
plt.tight_layout()
plt.show()

### 2.6 Visual EDA — Multivariate Analysis

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
corr_cols = numeric_cols + [target]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap (Numeric Features + Target)', fontsize=14)
plt.tight_layout()
plt.show()

# Show top correlations with target
target_corr = corr_matrix[target].drop(target).sort_values(key=abs, ascending=False)
print('Correlations with cost_overrun_pct (sorted by magnitude):')
print(target_corr.round(3))

In [ ]:
# Pair plot of top correlated features
top_features = target_corr.head(5).index.tolist()
pair_cols = top_features + [target]

g = sns.pairplot(df[pair_cols].sample(1000, random_state=42), diag_kind='kde',
                 plot_kws={'alpha': 0.3, 's': 15})
g.figure.suptitle('Pair Plot: Top 5 Correlated Features + Target', y=1.02, fontsize=14)
plt.show()

### 2.7 Key Findings from EDA

- **Target distribution:** `cost_overrun_pct` is roughly normally distributed, centered around 3-5%, with a range from about -15% to +55%
- **Strongest predictors:** `num_change_orders`, `risk_score`, and `complexity_score` show the highest correlation with cost overruns
- **Negative predictors:** Higher `contractor_rating` and `team_experience_years` are associated with lower overruns
- **No missing values:** The dataset is clean with no null entries
- **Balanced categories:** Project types and locations are reasonably well-distributed across the dataset

---
## 3. Data Preparation

### 3.1 Data Cleaning

In [ ]:
# Create a working copy for modeling
data = df.copy()

# Drop columns that would cause data leakage or are not useful
drop_cols = [
    'project_id', 'actual_cost_cad', 'actual_duration_weeks',
    'schedule_delay_pct', 'on_budget', 'on_schedule', 'data_source'
]
data = data.drop(columns=drop_cols)
print(f'After dropping leakage/ID columns: {data.shape}')

# Check for rows with zero or negative estimated_cost_cad
bad_rows = (data['estimated_cost_cad'] <= 0).sum()
print(f'Rows with zero/negative estimated cost: {bad_rows}')
if bad_rows > 0:
    data = data[data['estimated_cost_cad'] > 0]
    print(f'After removing: {data.shape}')

# Check for outliers using IQR on the target
Q1 = data[target].quantile(0.25)
Q3 = data[target].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = ((data[target] < lower) | (data[target] > upper)).sum()
print(f'\nTarget outliers (IQR method): {outliers} ({outliers/len(data)*100:.1f}%)')
print(f'IQR range: [{lower:.1f}%, {upper:.1f}%]')
# We keep the outliers — they represent real extreme overrun scenarios

### 3.2 Feature Engineering

In [ ]:
# One-hot encode categorical features
cat_features = ['project_type', 'location']
data_encoded = pd.get_dummies(data, columns=cat_features, drop_first=False, dtype=int)

print(f'Shape after one-hot encoding: {data_encoded.shape}')
print(f'New columns from encoding: {data_encoded.shape[1] - data.shape[1] + len(cat_features)}')

In [ ]:
# Separate features and target
X = data_encoded.drop(columns=[target])
y = data_encoded[target]

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature names ({X.shape[1]}):')
print(X.columns.tolist())

In [ ]:
# Standardize numeric features (z-score normalization)
from sklearn.preprocessing import StandardScaler

# Identify numeric (non-encoded) columns to scale
numeric_to_scale = [
    'sqft', 'floors', 'num_units', 'planned_duration_weeks',
    'estimated_cost_cad', 'contractor_rating', 'complexity_score',
    'team_experience_years', 'num_change_orders', 'weather_risk_factor',
    'material_volatility', 'num_subcontractors', 'budget_pressure',
    'risk_score', 'risk_score_original'
]

scaler = StandardScaler()
X[numeric_to_scale] = scaler.fit_transform(X[numeric_to_scale])

print('Standardization applied to numeric features.')
X[numeric_to_scale].describe().round(2)

### 3.3 Post-Cleaning EDA

After feature engineering, we verify the prepared dataset.

In [ ]:
# Verify no missing values after processing
print(f'Missing values in features: {X.isnull().sum().sum()}')
print(f'Missing values in target: {y.isnull().sum()}')
print(f'Final dataset size: {X.shape[0]} samples, {X.shape[1]} features')

# Check feature distributions after scaling
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_to_scale):
    axes[i].hist(X[col], bins=30, color='teal', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col} (scaled)', fontsize=9)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Feature Distributions After Standardization', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# 80/20 train-test split with fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nTarget distribution:')
print(f'  Train mean: {y_train.mean():.2f}%, std: {y_train.std():.2f}%')
print(f'  Test  mean: {y_test.mean():.2f}%, std: {y_test.std():.2f}%')

---
## 4. Model Building

We train **six different regression algorithms** to compare their performance on the cost overrun prediction task. This allows us to identify which approach works best for our specific dataset.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define models to compare
models = {
    'Ridge Regression': Ridge(alpha=10.0, random_state=42),
    'Lasso Regression': Lasso(alpha=0.1, random_state=42),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'SVR (RBF kernel)': SVR(kernel='rbf', C=10.0, epsilon=0.1),
}

print(f'Training {len(models)} models...')
print(f'Training samples: {X_train.shape[0]}, Features: {X_train.shape[1]}')

In [ ]:
import time

# Train all models and collect predictions
results = {}
trained_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    results[name] = {
        'Train MAE': mean_absolute_error(y_train, y_train_pred),
        'Test MAE': mean_absolute_error(y_test, y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train R2': r2_score(y_train, y_train_pred),
        'Test R2': r2_score(y_test, y_test_pred),
        'Train Time (s)': train_time,
    }
    trained_models[name] = model
    print(f'  {name}: Test R2={results[name]["Test R2"]:.4f}, '
          f'Test MAE={results[name]["Test MAE"]:.3f}%, '
          f'Time={train_time:.2f}s')

print('\nAll models trained.')

---
## 5. Model Evaluation

### 5.1 Performance Comparison

In [ ]:
# Summary table of all model results
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df.sort_values('Test R2', ascending=False)

In [ ]:
# Visual comparison of model performance
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_names = list(results.keys())
colors = ['#2563EB', '#16A34A', '#9333EA', '#E11D48', '#D97706', '#0891B2']

# R2 Score comparison
train_r2 = [results[m]['Train R2'] for m in model_names]
test_r2 = [results[m]['Test R2'] for m in model_names]
x_pos = np.arange(len(model_names))
axes[0].bar(x_pos - 0.2, train_r2, 0.35, label='Train', color='lightblue', edgecolor='steelblue')
axes[0].bar(x_pos + 0.2, test_r2, 0.35, label='Test', color='steelblue', edgecolor='navy')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
axes[0].set_title('R-squared Score')
axes[0].set_ylabel('R2')
axes[0].legend()

# MAE comparison
train_mae = [results[m]['Train MAE'] for m in model_names]
test_mae = [results[m]['Test MAE'] for m in model_names]
axes[1].bar(x_pos - 0.2, train_mae, 0.35, label='Train', color='#DCFCE7', edgecolor='green')
axes[1].bar(x_pos + 0.2, test_mae, 0.35, label='Test', color='green', edgecolor='darkgreen')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
axes[1].set_title('Mean Absolute Error (%)')
axes[1].set_ylabel('MAE')
axes[1].legend()

# RMSE comparison
train_rmse = [results[m]['Train RMSE'] for m in model_names]
test_rmse = [results[m]['Test RMSE'] for m in model_names]
axes[2].bar(x_pos - 0.2, train_rmse, 0.35, label='Train', color='#FEE2E2', edgecolor='salmon')
axes[2].bar(x_pos + 0.2, test_rmse, 0.35, label='Test', color='salmon', edgecolor='darkred')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
axes[2].set_title('Root Mean Squared Error (%)')
axes[2].set_ylabel('RMSE')
axes[2].legend()

plt.suptitle('Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs. Predicted scatter plots for each model
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    axes[i].scatter(y_test, y_pred, alpha=0.3, s=10, color=colors[i])
    # Perfect prediction line
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[i].plot(lims, lims, 'k--', linewidth=1, alpha=0.5)
    axes[i].set_xlabel('Actual Cost Overrun %')
    axes[i].set_ylabel('Predicted Cost Overrun %')
    axes[i].set_title(f'{name}\nR2={results[name]["Test R2"]:.4f}', fontsize=10)

plt.suptitle('Actual vs. Predicted (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Hyperparameter Tuning

We perform hyperparameter tuning on the best-performing model using cross-validated grid search.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Identify the best model from initial comparison
best_model_name = results_df['Test R2'].idxmax()
print(f'Best initial model: {best_model_name} (Test R2 = {results_df.loc[best_model_name, "Test R2"]:.4f})')
print(f'\nPerforming hyperparameter tuning...')

In [ ]:
# Hyperparameter tuning for Gradient Boosting (typically the strongest)
gb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)

gb_grid.fit(X_train, y_train)

print(f'Best Gradient Boosting parameters: {gb_grid.best_params_}')
print(f'Best CV R2: {gb_grid.best_score_:.4f}')

# Evaluate tuned model on test set
gb_tuned = gb_grid.best_estimator_
y_pred_tuned = gb_tuned.predict(X_test)

print(f'\nTuned Gradient Boosting on test set:')
print(f'  MAE:  {mean_absolute_error(y_test, y_pred_tuned):.4f}%')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.4f}%')
print(f'  R2:   {r2_score(y_test, y_pred_tuned):.4f}')

In [ ]:
# Also tune Ridge Regression (our production model)
ridge_param_grid = {
    'alpha': [0.1, 1.0, 5.0, 10.0, 50.0, 100.0],
}

ridge_grid = GridSearchCV(
    Ridge(random_state=42),
    ridge_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)

ridge_grid.fit(X_train, y_train)

print(f'Best Ridge alpha: {ridge_grid.best_params_}')
print(f'Best CV R2: {ridge_grid.best_score_:.4f}')

ridge_tuned = ridge_grid.best_estimator_
y_pred_ridge = ridge_tuned.predict(X_test)

print(f'\nTuned Ridge on test set:')
print(f'  MAE:  {mean_absolute_error(y_test, y_pred_ridge):.4f}%')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}%')
print(f'  R2:   {r2_score(y_test, y_pred_ridge):.4f}')

### 5.3 Feature Importance Analysis

In [ ]:
# Feature importance from Gradient Boosting (best model)
importances = gb_tuned.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

# Show top 15 features
plt.figure(figsize=(10, 7))
feat_imp.head(15).plot(kind='barh', color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances (Gradient Boosting)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 most important features:')
for feat, imp in feat_imp.head(10).items():
    print(f'  {feat}: {imp:.4f}')

In [ ]:
# Ridge Regression coefficients (interpretable linear model)
ridge_coefs = pd.Series(ridge_tuned.coef_, index=X.columns)
ridge_coefs_sorted = ridge_coefs.abs().sort_values(ascending=False)

plt.figure(figsize=(10, 7))
top_coefs = ridge_coefs[ridge_coefs_sorted.head(15).index].sort_values()
colors_bar = ['green' if v > 0 else 'red' for v in top_coefs]
top_coefs.plot(kind='barh', color=colors_bar)
plt.xlabel('Coefficient Value')
plt.title('Top 15 Ridge Regression Coefficients\n(Green = increases overrun, Red = decreases overrun)')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

### 5.4 Residual Analysis

In [ ]:
# Residual plots for the tuned Gradient Boosting model
residuals = y_test - y_pred_tuned

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs. predicted
axes[0].scatter(y_pred_tuned, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Cost Overrun %')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs. Predicted')

# Residual distribution
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution (mean={residuals.mean():.3f})')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals')

plt.suptitle('Residual Analysis (Tuned Gradient Boosting)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.5 Final Model Summary

In [ ]:
# Final comparison summary
print('=' * 70)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 70)

# Add tuned models to comparison
final_results = results_df[['Test MAE', 'Test RMSE', 'Test R2']].copy()
final_results.loc['Gradient Boosting (Tuned)'] = [
    mean_absolute_error(y_test, y_pred_tuned),
    np.sqrt(mean_squared_error(y_test, y_pred_tuned)),
    r2_score(y_test, y_pred_tuned)
]
final_results.loc['Ridge (Tuned)'] = [
    mean_absolute_error(y_test, y_pred_ridge),
    np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    r2_score(y_test, y_pred_ridge)
]

print(final_results.sort_values('Test R2', ascending=False).round(4).to_string())
print('\n' + '=' * 70)

best = final_results['Test R2'].idxmax()
print(f'\nBest model: {best}')
print(f'  Test R2:   {final_results.loc[best, "Test R2"]:.4f}')
print(f'  Test MAE:  {final_results.loc[best, "Test MAE"]:.4f}%')
print(f'  Test RMSE: {final_results.loc[best, "Test RMSE"]:.4f}%')
print(f'\nNote: In our production system, we use Ridge Regression because it')
print(f'offers the best balance of interpretability, speed, and performance')
print(f'for real-time cost estimation during live demos.')

---
## 6. Model Deployment and Monitoring (Optional)

Our cost estimation model is deployed as part of the **Industry AI Flow** web application:

- **Backend:** The trained Ridge Regression model is served via a FastAPI REST endpoint at `/api/v1/cost-estimation/predict`
- **Frontend:** A Next.js web interface provides an interactive form where users enter project parameters (sqft, type, location, risk factors) and receive real-time cost overrun predictions with confidence intervals
- **Monitoring:** API request latency, prediction distributions, and LLM usage are tracked via Prometheus metrics
- **Model artifact:** The trained model is serialized as a JSON file containing weights, normalization parameters, and evaluation metrics, enabling easy versioning and retraining

The system runs on local hardware (Mac Studio M1 Max or Windows + RTX 5060) for the Capstone Showcase demo.

---
## 7. Conclusion

In this notebook, we followed the CRISP-DM methodology to develop a machine learning model for predicting construction cost overruns.

**Key findings:**
- We compared **six regression algorithms**: Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, and SVR
- **Gradient Boosting** achieved the best test performance after hyperparameter tuning
- **Ridge Regression** was selected for production deployment due to its balance of interpretability, speed, and strong performance
- The most important predictors of cost overruns are: `num_change_orders`, `risk_score`, `contractor_rating`, and `team_experience_years`
- These findings align with domain expertise from our construction industry team member: projects with more change orders and higher risk scores tend to overrun their budgets, while experienced teams and higher-rated contractors help keep projects on budget